In [4]:
from sqlalchemy import create_engine
import pandas as pd

# Replace username, password, host, and database_name with your actual MySQL config
DB_USER = 'root'
DB_PASSWORD = 'Manoj1297%40'
DB_HOST = '127.0.0.1'
DB_NAME = 'tmdb'

# Create connection engine using PyMySQL driver
engine = create_engine(
    f'mysql+pymysql://{DB_USER}:{DB_PASSWORD}@{DB_HOST}/{DB_NAME}'
)
#1. List all movies in the Action genre released after 2015, showing title and release date.

# 1. Store your SQL query as a multi-line string
query = """
    SELECT m.title, m.release_date
    FROM movies m
    JOIN movie_genres mg ON m.movie_id = mg.movie_id
    JOIN genres g ON mg.genre_id = g.genre_id
    WHERE g.genre_name = 'Action' AND m.release_date > '2015-12-31'
    ORDER BY m.release_date DESC;
"""

# 2. Execute it using pd.read_sql and your database engine
df_result = pd.read_sql(query, engine)

# 3. Display the resulting DataFrame
display(df_result.head(10))

,title,release_date
0,Mortal Kombat II,2026-05-06
1,Predator: Badlands,2025-11-05
2,The Fantastic 4: First Steps,2025-07-23
3,Superman,2025-07-09
4,F1,2025-06-25
5,Jurassic World Rebirth,2025-06-23
6,How to Train Your Dragon,2025-06-06
7,Ballerina,2025-06-04
8,Mission: Impossible - The Final Reckoning,2025-05-17
9,Thunderbolts*,2025-04-30


In [5]:
import pandas as pd
#2. Find the top 10 highest-grossing movies along with the genre(s) each belongs to.

query_2 = """
    SELECT m.title, m.revenue, GROUP_CONCAT(g.genre_name SEPARATOR ', ') AS genres
    FROM movies m
    JOIN movie_genres mg ON m.movie_id = mg.movie_id
    JOIN genres g ON mg.genre_id = g.genre_id
    WHERE m.revenue > 0
    GROUP BY m.movie_id, m.title, m.revenue
    ORDER BY m.revenue DESC
    LIMIT 10;
"""
df_2 = pd.read_sql(query_2, engine)
display(df_2)

,title,revenue,genres
0,Avatar,2.923706e+09,"Adventure, Action, Science Fiction"
1,Avengers: Endgame,2.799439e+09,"Action, Adventure, Science Fiction"
2,Avatar: The Way of Water,2.334485e+09,"Adventure, Action, Science Fiction"
3,Titanic,2.264162e+09,"Romance, Drama"
4,Star Wars: The Force Awakens,2.068224e+09,"Action, Science Fiction, Adventure"
5,Avengers: Infinity War,2.052415e+09,"Science Fiction, Action, Adventure"
6,Spider-Man: No Way Home,1.921207e+09,"Adventure, Action, Science Fiction"
7,Zootopia 2,1.868209e+09,"Family, Adventure, Animation, Comedy, Mystery"
8,Inside Out 2,1.698864e+09,"Animation, Adventure, Comedy, Family"
9,Jurassic World,1.671537e+09,"Adventure, Science Fiction, Thriller"


In [6]:
query_3 = """
    SELECT 
        g.genre_name, 
        AVG(CASE WHEN m.budget > 0 THEN m.budget END) AS avg_budget,
        AVG(CASE WHEN m.revenue > 0 THEN m.revenue END) AS avg_revenue
    FROM genres g
    JOIN movie_genres mg ON g.genre_id = mg.genre_id
    JOIN movies m ON mg.movie_id = m.movie_id
    GROUP BY g.genre_id, g.genre_name
    ORDER BY avg_revenue DESC;
"""
df_3 = pd.read_sql(query_3, engine)
display(df_3)

,genre_name,avg_budget,avg_revenue
0,Adventure,1.106736e+08,4.003113e+08
1,Animation,8.763551e+07,3.718357e+08
2,Family,8.790630e+07,3.533691e+08
3,Science Fiction,9.672225e+07,3.336488e+08
4,Action,9.422751e+07,3.084532e+08
5,Fantasy,8.864007e+07,3.027060e+08
6,Comedy,5.450018e+07,2.215264e+08
7,Music,4.281702e+07,2.015662e+08
8,Romance,3.693970e+07,1.748110e+08
9,Thriller,5.121405e+07,1.680042e+08


In [7]:
query_4 = """
    SELECT actor_name, COUNT(DISTINCT movie_id) AS movie_count
    FROM `cast`
    GROUP BY actor_name
    ORDER BY movie_count DESC
    LIMIT 10;
"""
df_4 = pd.read_sql(query_4, engine)
display(df_4)

,actor_name,movie_count
0,Samuel L. Jackson,39
1,Robert De Niro,38
2,Brad Pitt,38
3,Johnny Depp,37
4,Tom Hanks,33
5,Scarlett Johansson,32
6,Willem Dafoe,32
7,Mark Wahlberg,32
8,Tom Cruise,31
9,Morgan Freeman,31


In [8]:
query_5 = """
    SELECT person_name AS director_name, COUNT(DISTINCT movie_id) AS movie_count
    FROM crew
    WHERE job = 'Director'
    GROUP BY person_id, person_name
    HAVING movie_count > 3
    ORDER BY movie_count DESC;
"""
df_5 = pd.read_sql(query_5, engine)
display(df_5)

,director_name,movie_count
0,Steven Spielberg,27
1,Tim Burton,19
2,Ridley Scott,19
3,Martin Scorsese,16
4,Robert Zemeckis,15
...,...,...
212,Robert Eggers,4
213,Ryan Coogler,4
214,Andy Muschietti,4
215,Wes Ball,4


In [9]:
query_6 = """
    SELECT keyword_name, COUNT(movie_id) AS keyword_frequency
    FROM movie_keywords
    GROUP BY keyword_id, keyword_name
    ORDER BY keyword_frequency DESC
    LIMIT 10;
"""
df_6 = pd.read_sql(query_6, engine)
display(df_6)

,keyword_name,keyword_frequency
0,based on novel or book,449
1,sequel,416
2,duringcreditsstinger,296
3,aftercreditsstinger,242
4,murder,176
5,based on true story,166
6,new york city,161
7,villain,160
8,3d animation,155
9,based on comic,154


In [10]:
query_7 = """
    SELECT title, budget, revenue
    FROM movies
    WHERE budget > (SELECT AVG(budget) FROM movies WHERE budget > 0)
      AND revenue < (SELECT AVG(revenue) FROM movies WHERE revenue > 0)
    ORDER BY budget DESC;
"""
df_7 = pd.read_sql(query_7, engine)
display(df_7.head(10))

,title,budget,revenue
0,The Marvels,274800000.0,206136825.0
1,Red One,250000000.0,185700759.0
2,Wake Up Dead Man: A Knives Out Mystery,210000000.0,4000000.0
3,Wonder Woman 1984,200000000.0,169601036.0
4,Luca,200000000.0,51074773.0
5,Green Lantern,200000000.0,219851172.0
6,Mulan,200000000.0,69965374.0
7,Onward,200000000.0,141888276.0
8,Jungle Cruise,200000000.0,220889446.0
9,The Gray Man,200000000.0,454023.0


In [11]:
query_8 = """
    SELECT DISTINCT c.actor_name, m.title
    FROM `cast` c
    JOIN crew cr ON c.movie_id = cr.movie_id
    JOIN movies m ON c.movie_id = m.movie_id
    WHERE cr.job = 'Director' AND cr.person_name = 'Christopher Nolan';
"""
df_8 = pd.read_sql(query_8, engine)
display(df_8.head(10))

,actor_name,title
0,Matthew McConaughey,Interstellar
1,Anne Hathaway,Interstellar
2,Michael Caine,Interstellar
3,Jessica Chastain,Interstellar
4,Casey Affleck,Interstellar
5,Wes Bentley,Interstellar
6,Topher Grace,Interstellar
7,Mackenzie Foy,Interstellar
8,Ellen Burstyn,Interstellar
9,John Lithgow,Interstellar


In [12]:
query_9 = """
    SELECT g.genre_name, COUNT(m.movie_id) AS movie_count, AVG(m.vote_average) AS avg_rating
    FROM genres g
    JOIN movie_genres mg ON g.genre_id = mg.genre_id
    JOIN movies m ON mg.movie_id = m.movie_id
    WHERE m.vote_count > 0
    GROUP BY g.genre_id, g.genre_name
    HAVING movie_count >= 20
    ORDER BY avg_rating DESC
    LIMIT 1;
"""
df_9 = pd.read_sql(query_9, engine)
display(df_9)

,genre_name,movie_count,avg_rating
0,War,85,7.357576


In [13]:
query_10 = """
    SELECT m.title, COUNT(mk.keyword_id) AS keyword_count
    FROM movies m
    JOIN movie_keywords mk ON m.movie_id = mk.movie_id
    GROUP BY m.movie_id, m.title
    ORDER BY keyword_count DESC
    LIMIT 10;
"""
df_10 = pd.read_sql(query_10, engine)
display(df_10)

,title,keyword_count
0,Smile 2,98
1,Taken 3,70
2,Silent Hill,63
3,Twelve Monkeys,57
4,War for the Planet of the Apes,54
5,The Shining,52
6,Dawn of the Planet of the Apes,51
7,How to Train Your Dragon,50
8,How to Train Your Dragon: The Hidden World,49
9,Pacific Rim: Uprising,49
